# Generate fixed Hindi speech audio for the Hindi Adult CI Auditory Training app

This notebook uses AI4Bharat's gated `ai4bharat/indic-parler-tts` model. The dependency setup below pins a compatible Transformers/Hub pair and explicitly restores a modern protobuf version after installing Parler-TTS, because Colab's preinstalled packages can otherwise be downgraded.

**Run the installation cell first, then restart the Colab runtime. After restarting, continue with the authentication cell.** If installation reports a fatal error, stop and share the complete output rather than running further repair commands.

Before running, accept the access conditions at [Hugging Face](https://huggingface.co/ai4bharat/indic-parler-tts). Never paste your token into chat or commit it to the repository. Review every generated recording with a Hindi-speaking clinician before clinical use; this is not a validated normative stimulus set.


In [ ]:
# Install in a controlled order. Parler-TTS may otherwise downgrade protobuf.
%pip install -q --upgrade pip
%pip install -q "huggingface_hub==0.36.2" "transformers==4.46.1" "tokenizers==0.20.3" sentencepiece accelerate soundfile requests
%pip install -q --no-deps git+https://github.com/huggingface/parler-tts.git
%pip install -q --upgrade "protobuf>=6.33.5,<7"

print('Installation commands finished. Now use Runtime → Restart session, then continue at the authentication cell.')


**Important after installation:** use **Runtime → Restart session**. The restart clears the Python state, which is expected. After it reconnects, continue at the Hugging Face authentication cell below; do not rerun the installation cell unless you intentionally want to reinstall the packages.


from huggingface_hub import notebook_login
notebook_login()


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


## 2. Load the current stimulus data and prepare a review batch
By default, this creates a **10-item preview** first. Listen to the WAVs and confirm Hindi pronunciation before generating the full set. Set `MAX_ITEMS = None` to process all unique speech strings. Environmental-sound module 1 is deliberately excluded because it needs sound effects, not spoken labels.

In [ ]:
import os, json, hashlib, zipfile, requests, torch, soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

DATA_URL = 'https://raw.githubusercontent.com/noorain04-web/Hindi-Adult-CI-Auditory-Training/main/data/training-data.json'
response = requests.get(DATA_URL, timeout=30)
response.raise_for_status()
data = response.json()

def collect_speech_texts(data):
    found = []
    seen = set()
    def add(value):
        if isinstance(value, str):
            value = value.strip()
            if value and value not in seen:
                seen.add(value)
                found.append(value)
    # Screening stimuli and response material
    screening = data.get('screening', {})
    for section in ('A', 'B'):
        for item in screening.get(section, []):
            if isinstance(item, dict):
                add(item.get('stimulus'))
                for option in item.get('options', []): add(option)
    for item in screening.get('C', []):
        if isinstance(item, dict):
            add(item.get('prompt')); add(item.get('answer'))
    for item in screening.get('D', []):
        if isinstance(item, dict): add(item.get('text'))
    # Modules 2-12. Module 1 is environmental sound effects and is excluded.
    for module_id, module in data.get('modules', {}).items():
        if str(module_id) == '1': continue
        for item in module.get('items', []):
            if isinstance(item, str):
                add(item)
            elif isinstance(item, (list, tuple)):
                for part in item[:2]: add(part)  # same/different pair; ignore boolean label
            elif isinstance(item, dict):
                add(item.get('stimulus'))
                for option in item.get('options', []): add(option)
                for key in ('text', 'question', 'answer', 'prompt'): add(item.get(key))
    return found

texts = collect_speech_texts(data)
MAX_ITEMS = 10  # Preview first; change to None to generate the complete set.
batch = texts if MAX_ITEMS is None else texts[:MAX_ITEMS]
print(f"Data version: {data.get('meta', {}).get('version', 'unknown')}")
print(f"Unique Hindi strings found: {len(texts)}; this run will generate: {len(batch)}")
print('\nPreview:')
for i, text in enumerate(batch[:20], 1): print(f'{i:03d}. {text}')


## 3. Load Indic Parler-TTS
The model card recommends Rohit or Divya for Hindi. This notebook uses Rohit's voice description consistently for a stable speaker across items. Voice quality and pronunciation still require human review.

In [ ]:
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

MODEL_ID = 'ai4bharat/indic-parler-tts'
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device.startswith('cuda') else torch.float32
print('Device:', device, '| dtype:', dtype)
model = ParlerTTSForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

VOICE_DESCRIPTION = ("Rohit's voice speaks Hindi in a neutral, clear, adult male voice at a moderate and steady pace. "
                     "The pronunciation is precise and natural, with very clear audio, close microphone placement, "
                     "and no background noise.")
description_inputs = description_tokenizer(VOICE_DESCRIPTION, return_tensors='pt').to(device)
print('Model loaded. Review the generated preview before making the full pack.')


## 4. Generate WAV files and a text-to-file manifest
Files are named by a stable hash of the exact Hindi text. Existing files are skipped so a disconnected Colab session can be resumed. If a pronunciation is wrong, edit the text or manually replace the WAV after review; do not silently accept incorrect clinical stimuli.

In [ ]:
OUT_DIR = Path('/content/hindi_ci_audio')
AUDIO_DIR = OUT_DIR / 'audio' / 'hi-IN'
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUT_DIR / 'audio-manifest.json'
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
else:
    manifest = {'model': MODEL_ID, 'speaker_description': VOICE_DESCRIPTION, 'items': {}}

for i, text in enumerate(batch, 1):
    key = hashlib.sha256(text.encode('utf-8')).hexdigest()[:16]
    filename = f'{key}.wav'
    filepath = AUDIO_DIR / filename
    if text in manifest['items'] and filepath.exists():
        print(f'[{i}/{len(batch)}] already exists: {text}')
        continue
    prompt_inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.inference_mode():
        generation = model.generate(
            input_ids=description_inputs.input_ids,
            attention_mask=description_inputs.attention_mask,
            prompt_input_ids=prompt_inputs.input_ids,
            prompt_attention_mask=prompt_inputs.attention_mask,
        )
    audio = generation.detach().float().cpu().numpy().squeeze()
    sf.write(str(filepath), audio, model.config.sampling_rate, subtype='PCM_16')
    manifest['items'][text] = f'audio/hi-IN/{filename}'
    MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'[{i}/{len(batch)}] saved: {text}')

print(f"\nSaved {len(manifest['items'])} text-to-audio entries in {OUT_DIR}")


## 5. Listen to a sample and export the files
Check the sample for correct pronunciation, vowel length, aspiration, natural pauses, and intelligibility. For the full pack, change `MAX_ITEMS = None` in section 2 and rerun sections 2–5. Download the ZIP and review the WAVs before adding them to the website.

In [ ]:
sample_text = next(iter(manifest['items'])) if manifest['items'] else None
if sample_text:
    sample_path = OUT_DIR / manifest['items'][sample_text]
    print('Sample text:', sample_text)
    sample_audio, sample_rate = sf.read(sample_path)
    display(Audio(sample_audio, rate=sample_rate))

zip_path = Path('/content/hindi_ci_audio_pack.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUT_DIR.rglob('*'):
        if path.is_file(): zf.write(path, path.relative_to(OUT_DIR))
print('ZIP ready:', zip_path)
from google.colab import files
files.download(str(zip_path))


## Licensing and clinical review
- Model card: [AI4Bharat Indic Parler-TTS](https://huggingface.co/ai4bharat/indic-parler-tts), listed under Apache-2.0; access is gated and requires accepting Hugging Face's conditions.
- Keep model attribution and review the current model/access terms before distributing the generated recordings. The model licence alone does not certify clinical validity or guarantee every generated utterance is correct.
- This notebook only generates speech audio. It does not validate the stimuli, calibrate presentation level, establish speech-in-noise SNR, or upload recordings to GitHub.
